In [12]:
#DESCRIPCIÓN: Junta todos los 18 archivos con las señales en 
import pandas as pd #sirve para leer y guardar Excel
import glob #sirve para buscar archivos que siguen un patrón
import os #sirve para construir rutas de carpetas y archivos

#carpteta donde se han guardado los excels del paso anterior
carpeta = r"C:\Users\guill\TFG\OptimizaciónSemiparamétrica"
#toma todos los archivos que se lleman senales_ (1 por cada base de datos=18)
archivos = glob.glob(os.path.join(carpeta, "senales_*.xlsx"))
#se crea una lista vacía
lista = []
#para cada uno de los 18 archivos, los lee y junta todo en una tabla ( las columnas siguen siendo las de antes y se tienen 789x18=14
for archivo in archivos:
    df = pd.read_excel(archivo)
    lista.append(df)
senales = pd.concat(lista, ignore_index=True)

#convierte la columna fecha a formato fecha por si acasp ( en nuestro caso ya estaba)
senales["Fecha"] = pd.to_datetime(senales["Fecha"])
#ordena la tabla para que se vean Fecha y Estrategia al lado
senales = senales.sort_values(["Fecha", "Estrategia"])
#crea el archivo para que se guarde en la carpeta
salida = os.path.join(carpeta, "senales_rf_historicas.xlsx")
senales.to_excel(salida, index=False)

In [14]:
#DESCRIPCIÓN: Se estandarizan las señales del archivo que recoge todas conjuntas
#El cálculo de las señales esatndarizadas se realiza restando la media de toda la columna y dividiendo entre la desviación típica de toda la columna también
senales["z_pred"] = senales.groupby("Fecha")["Pred_RF"].transform(
    lambda x: (x - x.mean()) / x.std()
)

senales["z_prob"] = senales.groupby("Fecha")["Prob_Relativa"].transform(
    lambda x: (x - x.mean()) / x.std()
)

senales["z_mom"] = senales.groupby("Fecha")["Momentum_3M"].transform(
    lambda x: (x - x.mean()) / x.std()
)
#añade 3 columnas más a las senales del apartado anterior
senales = senales.dropna(subset=["z_pred", "z_prob", "z_mom"])
#se crea un nuevo excel que recoja tamnbién esta información con toda la anterior
salida = os.path.join(carpeta, "senales_estandarizadas.xlsx")
senales.to_excel(salida, index=False)

In [247]:
#DESCRIPCIÓN: se crean las matrices que se necesitan luego

#se importan las librerías que se necesitan
import pandas as pd #sirve para leer excel y trabajar con tablas
import numpy as np #sirve para construir matrives numéricas
import os #sirce para trabajar con rutas de archivos
from scipy.optimize import minimize #se utilizara depués

#se utiliza el último archivo
carpeta = r"C:\Users\guill\TFG\OptimizaciónSemiparamétrica"
archivo = os.path.join(carpeta, "senales_estandarizadas.xlsx")
senales = pd.read_excel(archivo)


#Este bloque sirve para borrar aquellas fechas en las que no haya 18 estrategias
conteo = senales.groupby("Fecha")["Estrategia"].nunique() #se cuenta cuántas estrategias hay por fecha
n_estrategias = senales["Estrategia"].nunique() #cuenta cuantas estrategias hay en total
fechas_completas = conteo[conteo == n_estrategias].index  #se queda solo con aquellaas fechas que tengan las 18 estrategias (en nuestro caso no elimina ninguna)
senales = senales[senales["Fecha"].isin(fechas_completas)].copy()

#Crea las matrices, es decir, se crean 4 hojas  distintas ( 3 para las señales y una para los rendimientos del t+1) de manera que las columnas sean fecha y las 18 estrategias y las filas sean la fecha
z_pred = senales.pivot(index="Fecha", columns="Estrategia", values="z_pred")
z_prob = senales.pivot(index="Fecha", columns="Estrategia", values="z_prob")
z_mom = senales.pivot(index="Fecha", columns="Estrategia", values="z_mom")
rend = senales.pivot(index="Fecha", columns="Estrategia", values="Rendimiento_real_t1")

#Otra vez se comprueba que esta todas las fechas completas con las 18 estrategias ( en nuestro caso ya lo habiamos comprobado peor por si acaso9
fechas_comunes = (
    z_pred.dropna().index
    .intersection(z_prob.dropna().index)
    .intersection(z_mom.dropna().index)
    .intersection(rend.dropna().index)
)

#hace que todas las matrices tengan las misma fechas y en el mismo orden
z_pred = z_pred.loc[fechas_comunes]
z_prob = z_prob.loc[fechas_comunes]
z_mom = z_mom.loc[fechas_comunes]
rend = rend.loc[fechas_comunes]

#Z es la matriz tridimensional  (T meses x N estrategias x 3 señales)
Z = np.stack(
    [z_pred.values, z_prob.values, z_mom.values],
    axis=2
)
#matriz de rendimientos en t+1 ( T meses x N estrategias)
R = rend.values

#se guardan en T N y K cuantos meses estrategias y señales hay 
T, N, K = Z.shape

In [149]:
#DESCRIPCIÓN:se guarda el excel con las 4 matrices en un archivo con 4 hojas
#donde se va a guardar
salida_matrices = os.path.join(carpeta, "matrices_optimizacion.xlsx")
#te crea el archivo de 4 hojas que he explicado antes
with pd.ExcelWriter(salida_matrices, engine="openpyxl") as writer:
    z_pred.to_excel(writer, sheet_name="z_pred")
    z_prob.to_excel(writer, sheet_name="z_prob")
    z_mom.to_excel(writer, sheet_name="z_mom")
    rend.to_excel(writer, sheet_name="rendimientos")

In [249]:
#DESCRIPCIÓN: se crea la función que calcula los pesos
def calcular_pesos(theta, Z_t):
    #theta = vector (theta 1, theta 2, theta 3) que se bsca optimizar
    #Z_t= matriz NX3 de señales estandarizas en la fecha t

    #estrategias que hay en cada t, es decir, número de filas de Z_t
    N = Z_t.shape[0]

    #cálculo del benchmark equiponderado: w base es el vector que contiene los 18 pesos equiponderados
    w_base = np.ones(N) / N

    #se cal ula el ajuste, es decir, al cantidad que el modelo suma o resta al peos inicial de cada estartegia. cada componente es 1/N del vector que contiene cada señal por el peso de la señal en concreto
    ajuste = (1 / N) * (Z_t @ theta)

    #el peso se calcula conforme esya fórmula
    w = w_base + ajuste

    #rentricción 1: no hay pesos negativos
    w = np.maximum(w, 0)

    # Normalización
    if w.sum() == 0:
        w = w_base #si todos los pesos son cero entonces vuelve al benchmark
    else:
        w = w / w.sum()  #lo normalizamos

    return w

In [251]:
#DESCRIPCIÓN: se define la función de utilidad
gamma = 5  #se establece este valor para el parámetro gamma

def utilidad_crra(rp):
    #Utilidad CRRA 
    if 1 + rp <= 0:
        return -1e10
    return ((1 + rp) ** (1 - gamma)) / (1 - gamma) #fórmula de la utilidad

In [205]:
#DESCRIPCIÓN: CASO 1 (CON LA MEDIA) se define la función objetivo que se busca máximizar( -minimizar) + se pone límite a los parámetros
#EL LIMITE SE MODIFICA PARA VER VARIOS CASOS ( en el caso de no querer límite se quita )
def objetivo_penalizado(theta, Z_train, R_train):
    #theta = vector (theta 1, theta 2, theta 3) que se bsca optimizar
    #Z_train = valores de Z que se utilizan para entrenar (Z es la matriz tridimensional  (T meses x N estrategias x 3 señales))
    #R_train = valores de R que se utilizan para entrenar ( R es la matriz de rendimientos en t+1 ( T meses x N estrategias))p
    # Penalización si theta se sale del rango razonable
    #if np.any(np.abs(theta) > limite):
        #return 1e6 + np.sum(theta**2) #si theat supera el límite, se devuelve un valor muy alto que la función no vaya a minimizarse de ninguna manera

    rentabilidades = [] #se crea la lista vacia para guardar las rentabillidades

    for t in range(len(R_train)): #este bucle recorre todas las fechas del periodo de entrenamiento y en cada t calcula los pesos de la cartera y su rentabilidad
        w_t = calcular_pesos(theta, Z_train[t]) #se calcula los pesos en el tiempo t
        rp_t = np.sum(w_t * R_train[t]) #se calculan las rentabilidades
        rentabilidades.append(rp_t) #se añaden a la lista
    rentabilidades = np.array(rentabilidades) #convierte la lsita en array

    #para cada rentabilidad de la lista calcula la rentabilidad con la función definida antes 
    utilidades = np.array([
        utilidad_crra(rp) for rp in rentabilidades
    ])

    return -np.mean(utilidades) 

In [253]:
#DESCRIPCIÓN: CASO 2 (ROLLING WINDOW) se define la función objetivo 

from scipy.optimize import minimize #se importa para la función de minimizar

def rolling_window_optimization(Z, R, ventana=120,limite=1):
   
    #Z = matriz tridimensional de señales estandarizadas
    #R = matriz de rendimientos reales
    #ventana = número de meses usados para entrenar
    #limite = límite máximo permitido para theta

    thetas_optimos = [] #guarda el theta óptimo de cada ventana
    pesos_finales = [] #guarda los pesos asignados a las estrategias
    rentabilidades_cartera = [] #guarda la rentabilidad obtenida por la cartera
    fechas_prediccion = [] #guarda el instante temporal correspondiente

    theta_inicial = np.array([0, 0, 0]) #theta inicial

    #empieza el bucle
    for t in range(ventana, len(R)):

        # Periodo de entrenamiento: ventana móvil
        Z_train = Z[t-ventana:t]
        R_train = R[t-ventana:t]

        # Optimización de theta usando solo la ventana reciente
        resultado = minimize(
            objetivo_penalizado,
            theta_inicial,
            args=(Z_train, R_train, limite),
            method="Nelder-Mead"
        )

        theta_optimo = resultado.x #se guarda eñ theta optimo

        # señales disponibles en el momento t
        Z_t = Z[t]

        # pesos para invertir en el momento t
        w_t = calcular_pesos(theta_optimo, Z_t)

        # rentabilidad real obtenida en t
        rp_t = np.sum(w_t * R[t])

        # se guarda
        thetas_optimos.append(theta_optimo)
        pesos_finales.append(w_t)
        rentabilidades_cartera.append(rp_t)
        fechas_prediccion.append(t)

        # se usa el theta óptimo como punto inicial del siguiente mes
        theta_inicial = theta_optimo
        
#se devuelve
    return {
        "thetas_optimos": np.array(thetas_optimos),
        "pesos_finales": np.array(pesos_finales),
        "rentabilidades_cartera": np.array(rentabilidades_cartera),
        "fechas_prediccion": np.array(fechas_prediccion)
    }

In [255]:
#DESCRIPCIÓN: calcular intervalos para entrenar el modelo
corte = int(0.8 * T) #se coge el 80% de los datos para entrenar
#se separan las variables para entrenar el modelo y para testearlo
Z_train = Z[:corte]
R_train = R[:corte]
Z_test = Z[corte:]
R_test = R[corte:]

In [209]:
#DECRIPCIÓN: optimización de theta
theta_inicial = np.array([0.0, 0.0, 0.0]) #se parte de un vector nulo
#se minimiza la función objetivo, partiendo del theta inicial con el Metodo de nelder Mead
opt = minimize(
    fun=objetivo_penalizado,
    x0=theta_inicial,
    args=(Z_train, R_train),
    method="Nelder-Mead",
    options={
        "maxiter": 5000, 
        "xatol": 1e-8,
        "fatol": 1e-8,
        "disp": True
    }
)

theta_opt = opt.x #guarda el vector óptimo
#devuelve los valores
print("Theta óptimo:")
print(theta_opt)
print("Valor función objetivo:")
print(opt.fun)
print("Convergencia:")
print(opt.success)



Optimization terminated successfully.
         Current function value: 0.245553
         Iterations: 441
         Function evaluations: 956
Theta óptimo:
[ 5.63859027e+11 -9.72161129e+10 -4.78672759e+10]
Valor función objetivo:
0.2455531348839467
Convergencia:
True


In [ ]:
#DESCRIPCIÓN: se recogen los resultados del rolling windor con límite

resultados_rolling = rolling_window_optimization1(
    Z,
    R,
    ventana=120
)

thetas_rolling = resultados_rolling["thetas_optimos"]
pesos_rolling = resultados_rolling["pesos_finales"]
rentabilidades_rolling = resultados_rolling["rentabilidades_cartera"]

print("Thetas rolling:")
print(thetas_rolling)

print("Pesos rolling:")
print(pesos_rolling)

print("Rentabilidades de la cartera rolling:")
print(rentabilidades_rolling)

In [78]:
#DESCRIPCIÓN: se calculan los estadísticos de los valores obtenidos en el rollling window
#se cargan las librerias
import pandas as pd
import os
#se convierte a dataframe los valores obtenidos antes
thetas_rolling_df = pd.DataFrame(
    thetas_rolling,
    columns=["theta_pred", "theta_prob", "theta_mom"]
)

# Si tienes rend.index y usaste ventana = 120, añadimos fechas (esto es por si acaso, no hace flata)
ventana = 120

if "rend" in globals():
    fechas_rolling = rend.index[ventana:ventana + len(thetas_rolling_df)]
    thetas_rolling_df.index = fechas_rolling
    thetas_rolling_df.index.name = "Fecha"

# se calcula lo que queriamos
theta_resumen = thetas_rolling_df.describe().T

theta_resumen = theta_resumen[["mean", "std", "min", "max"]]

theta_resumen.columns = [
    "Media",
    "Desv. típica",
    "Mínimo",
    "Máximo"
]
#se imprime
print("Resumen de theta rolling:")
print(theta_resumen)



Resumen de theta rolling:
               Media  Desv. típica  Mínimo    Máximo
theta_pred  0.917019      0.380526    -1.0  1.000000
theta_prob  0.773768      0.584937    -1.0  1.000000
theta_mom  -0.269927      0.345662    -1.0  0.816229

Archivos guardados:
C:\Users\guill\TFG\OptimizaciónSemiparamétrica\resumen_thetas_rolling.xlsx
C:\Users\guill\TFG\OptimizaciónSemiparamétrica\thetas_rolling_completos.xlsx


In [211]:
#DESCRIPCIÓN:sacar las métricas del CASO del rolling (IGUAL QUE LA DE LA MEDIA PERO SE CAMBIAN COSAS)
def max_drawdown(r):
    acumulada = np.cumprod(1 + r)
    max_acum = np.maximum.accumulate(acumulada)
    drawdown = acumulada / max_acum - 1
    return drawdown.min()


def metricas(r):
    r = np.array(r)

    media_mensual = np.mean(r)
    vol_mensual = np.std(r)

    rent_anual = media_mensual * 12
    vol_anual = vol_mensual * np.sqrt(12)

    sharpe = rent_anual / vol_anual if vol_anual != 0 else np.nan

    return {
        "Rentabilidad mensual media": media_mensual,
        "Volatilidad mensual": vol_mensual,
        "Rentabilidad anualizada": rent_anual,
        "Volatilidad anualizada": vol_anual,
        "Sharpe": sharpe,
        "Rentabilidad acumulada": np.prod(1 + r) - 1,
        "Máximo drawdown": max_drawdown(r),
        "% meses positivos": np.mean(r > 0)
    }


# Rentabilidad de la cartera rolling
rent_rolling = resultados_rolling["rentabilidades_cartera"]

# Benchmark equiponderado desde el mismo punto temporal
ventana = 120
rent_benchmark_rolling = R[ventana:].mean(axis=1)

# Comprobación para asegurar que tienen la misma longitud
print("Longitud rolling:", len(rent_rolling))
print("Longitud benchmark:", len(rent_benchmark_rolling))

# Tabla de métricas
tabla_metricas_rolling = pd.DataFrame({
    "Cartera optimizada rolling": metricas(rent_rolling),
    "Benchmark equiponderado": metricas(rent_benchmark_rolling)
})

print(tabla_metricas_rolling)

# Guardar en Excel
tabla_metricas_rolling.to_excel(
    os.path.join(carpeta, "metricas_rolling.xlsx")
)

Longitud rolling: 669
Longitud benchmark: 669
                            Cartera optimizada rolling  \
Rentabilidad mensual media                    0.004462   
Volatilidad mensual                           0.022759   
Rentabilidad anualizada                       0.053538   
Volatilidad anualizada                        0.078838   
Sharpe                                        0.679089   
Rentabilidad acumulada                       15.452930   
Máximo drawdown                              -0.304014   
% meses positivos                             0.669656   

                            Benchmark equiponderado  
Rentabilidad mensual media                 0.002025  
Volatilidad mensual                        0.020308  
Rentabilidad anualizada                    0.024301  
Volatilidad anualizada                     0.070348  
Sharpe                                     0.345445  
Rentabilidad acumulada                     2.364789  
Máximo drawdown                           -0.385427  

In [215]:
#DESCRIPCIÓN: se define la función max drawdown y metricas para el caso de la media


def max_drawdown(r): #se calcula la mayor caída acumplada que sufre la cartera 
    acumulada = np.cumprod(1 + r) #rentabilidad acumulada
    max_acum = np.maximum.accumulate(acumulada) #calcula el maximo histórico alcanzado por la cartera hasta esa fecha
    drawdown = acumulada / max_acum - 1 #se calcula ciuanto ha caido la cartera respecto a su maximo anterior
    return drawdown.min()  #se devuelve la peor caída


def metricas(r): #se calculan varías métricas
    r = np.array(r)
    media_mensual = np.mean(r) #media menusal de las rentabilidades de todos los meses 
    vol_mensual = np.std(r) #desviación  típica de las rentabilidades
    rent_anual = media_mensual * 12 #mismos datos pero anuales
    vol_anual = vol_mensual * np.sqrt(12) 

    sharpe = rent_anual / vol_anual if vol_anual != 0 else np.nan #ratio sharpe
    
#se imprimen todos los resultados
    return {
        "Rentabilidad mensual media": media_mensual,
        "Volatilidad mensual": vol_mensual,
        "Rentabilidad anualizada": rent_anual,
        "Volatilidad anualizada": vol_anual,
        "Sharpe": sharpe,
        "Rentabilidad acumulada": np.prod(1 + r) - 1,
        "Máximo drawdown": max_drawdown(r),
        "% meses positivos": np.mean(r > 0)
    }


tabla_metricas = pd.DataFrame({
    "Cartera optimizada": metricas(rent_opt),
    "Benchmark equiponderado": metricas(rent_benchmark)
})

print(tabla_metricas)
#se guarda en excel
tabla_metricas.to_excel(
    os.path.join(carpeta, "metricas_nelder_mead.xlsx")
)

                            Cartera optimizada  Benchmark equiponderado
Rentabilidad mensual media            0.004929                 0.003974
Volatilidad mensual                   0.028706                 0.020814
Rentabilidad anualizada               0.059150                 0.047693
Volatilidad anualizada                0.099441                 0.072101
Sharpe                                0.594825                 0.661475
Rentabilidad acumulada                1.034963                 0.808349
Máximo drawdown                      -0.284208                -0.139713
% meses positivos                     0.632911                 0.651899


In [237]:
#DESCRIPCIÓN: aqui se obtienen los pesos del primer mes que no sabemos su rentabilidad
Z_ultima = Z[-3] #toma el último valor
ultima_fecha = rend.index[-3]
w_recomendado = calcular_pesos(theta_opt, Z_ultima) #utiliza la función de anres

#crea un excel con los pesos recomendados y crea una tabla que imprime
recomendacion = pd.DataFrame({
    "Estrategia": rend.columns,
    "Peso_recomendado": w_recomendado
})

recomendacion = recomendacion.sort_values(
    "Peso_recomendado",
    ascending=False
)

recomendacion.to_excel(
    os.path.join(carpeta, "recomendacion_mes_siguiente_nelder_mead.xlsx"),
    index=False
)

print("Última fecha disponible:", ultima_fecha)
print(recomendacion)

Última fecha disponible: 2025-09-01 00:00:00
              Estrategia  Peso_recomendado
9      Japan_Bolsa_Bonos          0.244028
10      Japan_Bolsa_Cash          0.162261
15       USA_Bolsa_Bonos          0.129229
6     Europe_Bolsa_Bonos          0.115545
4      Canada_Bolsa_Cash          0.092925
3     Canada_Bolsa_Bonos          0.089685
16        USA_Bolsa_Cash          0.087500
12        UK_Bolsa_Bonos          0.060547
7      Europe_Bolsa_Cash          0.018280
14         UK_Bonos_Cash          0.000000
13         UK_Bolsa_Cash          0.000000
0   Australia_Bonos_Cash          0.000000
11      Japan_Bonos_Cash          0.000000
1   Autralia_Bolsa_Bonos          0.000000
8      Europe_Bonos_Cash          0.000000
5      Canada_Bonos_Cash          0.000000
2    Autralia_Bolsa_Cash          0.000000
17        USA_Bonos_Cash          0.000000


In [233]:
#DESCRIPCIÓN:te calucula los pesos recomendados
# Última matriz de señales disponible
Z_ultima = Z[-3]

# Última fecha disponible en la base
ultima_fecha = rend.index[-3]

# Último theta estimado mediante rolling window
theta_ultimo = thetas_rolling[-3]

# Calcular pesos recomendados usando el último theta rolling
w_recomendado = calcular_pesos(theta_ultimo, Z_ultima)

# Crear tabla con pesos recomendados
recomendacion_rolling = pd.DataFrame({
    "Estrategia": rend.columns,
    "Peso_recomendado": w_recomendado
})

# Ordenar de mayor a menor peso
recomendacion_rolling = recomendacion_rolling.sort_values(
    "Peso_recomendado",
    ascending=False
)

# Guardar en Excel
recomendacion_rolling.to_excel(
    os.path.join(carpeta, "recomendacion_mes_siguiente_rolling.xlsx"),
    index=False
)

# Mostrar resultados
print("Última fecha disponible:", ultima_fecha)
print("Último theta rolling utilizado:")
print(theta_ultimo)

print("\nRecomendación para el mes siguiente con rolling:")
print(recomendacion_rolling)

Última fecha disponible: 2025-09-01 00:00:00
Último theta rolling utilizado:
[-0.99999649 -0.39759885  0.81622878]

Recomendación para el mes siguiente con rolling:
              Estrategia  Peso_recomendado
5      Canada_Bonos_Cash          0.104841
1   Autralia_Bolsa_Bonos          0.097731
8      Europe_Bonos_Cash          0.092537
14         UK_Bonos_Cash          0.091869
17        USA_Bonos_Cash          0.091537
0   Australia_Bonos_Cash          0.086899
2    Autralia_Bolsa_Cash          0.086899
13         UK_Bolsa_Cash          0.074288
11      Japan_Bonos_Cash          0.067676
12        UK_Bolsa_Bonos          0.049012
3     Canada_Bolsa_Bonos          0.042945
7      Europe_Bolsa_Cash          0.029316
4      Canada_Bolsa_Cash          0.028908
16        USA_Bolsa_Cash          0.027329
6     Europe_Bolsa_Bonos          0.014901
15       USA_Bolsa_Bonos          0.013311
10      Japan_Bolsa_Cash          0.000000
9      Japan_Bolsa_Bonos          0.000000


In [239]:
# Rendimientos reales de la última fecha disponible
R_ultima = R[-1]
print(R_ultima)

# Rentabilidad de la cartera recomendada en esa última fecha
rentabilidad_mes = np.sum(w_recomendado * R_ultima)

print("Rentabilidad de la cartera recomendada en la última fecha disponible:")
print(rentabilidad_mes)

[-0.02746888  0.03565061 -0.02746888  0.02385332  0.0082003  -0.01565302
  0.03190696  0.0200008  -0.01190616  0.02183152  0.00149442 -0.0203371
  0.04164397  0.01846996 -0.02317401  0.00815472 -0.00371874 -0.01187346]
Rentabilidad de la cartera recomendada en la última fecha disponible:
0.015773470716436665
